In [ ]:
using DataFrames, CSV, Dates, Statistics, ShiftedArrays, Plots, StatFiles, DataFramesMeta
import ShiftedArrays: lead, lag

In [ ]:
df = CSV.read("D:/PSID_SHELF/03_NLSY97/data/NLSY97.csv", DataFrame)


In [ ]:
# Define a regular expression pattern for replacement
pattern = r"_XRND"

# Define a function to apply the renaming
new_name(name) = Symbol(replace(name, pattern => ""))

# Rename columns using the rename! function
rename!(df, new_name.(names(df)))


In [ ]:
function rename_variables(df)
    counter = 1  # Initialize counter
    for year in 1994:2022
        weeks = year == 2022 ? 40 : (year in [1994, 2000, 2005, 2011, 2016] ? 53 : 52)
        start_week = year == 1994 ? 6 : 1
        
        for week in start_week:weeks
            oldvar = Symbol("EMP_STATUS_$(year)_$(lpad(week, 2, '0'))")
            newvar = Symbol("EMPSTATUS_$counter")
            if hasproperty(df, oldvar)  # Check if the variable exists
                rename!(df, oldvar => newvar)
            end
            counter += 1
        end
    end
    return df
end

# Call the function with df and store the result in df_re
df = rename_variables(df)
show(df)


In [ ]:
for year in [1997:2011; 2013:2:2021]
    colname = Symbol("INTERVIEWER_PUBID_$year")
    if hasproperty(df, colname)
        select!(df, Not(colname))
    end
end
df

In [ ]:
df1 = stack(df, Between(:EMPSTATUS_1, :EMPSTATUS_1496), [:PUBID_1997], variable_name=:week, value_name="EMPSTATUS_")
sort!(df1, :PUBID_1997)

In [ ]:
df1[!, :week] = replace.(df1[!, :week], r"EMPSTATUS_" => "")
df1[!, :week] = replace.(df1[!, :week], "_" => "")
df1[!, :week] = parse.(Int, df1[!, :week])
sort!(df1, :PUBID_1997)

In [ ]:
# Generate date based on the week variable
df1.date = [Date(1994, 2, 12) + Day(7 * (week - 1)) for week in df1.week]

# Extract year, month, and week from the date
df1.year = year.(df1.date)
df1.month = month.(df1.date)
df1.Week = Dates.week.(df1.date)
sort!(df1, :PUBID_1997)


In [ ]:

# Create df3 with columns from EMPSTATUS_1 to EMPSTATUS_1496
df3 = select(df, Between(:EMPSTATUS_1, :EMPSTATUS_1496))

# Subtract df3 from df2 to get the remaining columns
df2 = select(df, Not(names(df3)))


In [ ]:
df_long = rightjoin(df1, df2, on=:PUBID_1997)


In [ ]:
# Generate weight
df_long[!, :weight] = zeros(length(df_long[!, :PUBID_1997]))

# Sort by PUBID_1997
sort!(df_long, :PUBID_1997)

# Create a dictionary to map years to weight columns
weight_columns = Dict(
    1994 => :SAMPLING_PANEL_WEIGHT_1997,
    1995 => :SAMPLING_PANEL_WEIGHT_1997,
    1996 => :SAMPLING_PANEL_WEIGHT_1997,
    1997 => :SAMPLING_PANEL_WEIGHT_1997,
    1998 => :SAMPLING_PANEL_WEIGHT_1998,
    1999 => :SAMPLING_PANEL_WEIGHT_1999,
    2000 => :SAMPLING_PANEL_WEIGHT_2000,
    2001 => :SAMPLING_PANEL_WEIGHT_2001,
    2002 => :SAMPLING_PANEL_WEIGHT_2002,
    2003 => :SAMPLING_PANEL_WEIGHT_2003,
    2004 => :SAMPLING_PANEL_WEIGHT_2004,
    2005 => :SAMPLING_PANEL_WEIGHT_2005,
    2006 => :SAMPLING_PANEL_WEIGHT_2006,
    2007 => :SAMPLING_PANEL_WEIGHT_2007,
    2008 => :SAMPLING_PANEL_WEIGHT_2008,
    2009 => :SAMPLING_PANEL_WEIGHT_2009,
    2010 => :SAMPLING_PANEL_WEIGHT_2010,
    2011 => :SAMPLING_PANEL_WEIGHT_2011,
    2012 => :SAMPLING_PANEL_WEIGHT_2013,
    2013 => :SAMPLING_PANEL_WEIGHT_2013,
    2014 => :SAMPLING_PANEL_WEIGHT_2015,
    2015 => :SAMPLING_PANEL_WEIGHT_2015,
    2016 => :SAMPLING_PANEL_WEIGHT_2017,
    2017 => :SAMPLING_PANEL_WEIGHT_2017,
    2018 => :SAMPLING_PANEL_WEIGHT_2019,
    2019 => :SAMPLING_PANEL_WEIGHT_2019,
    2020 => :SAMPLING_PANEL_WEIGHT_2021,
    2021 => :SAMPLING_PANEL_WEIGHT_2021
)

# Assign weights based on the year and PUBID_1997
for i in 1:length(df_long[!, :PUBID_1997])
    year_value = year(df_long[!, :date][i])
    if haskey(weight_columns, year_value)
        df_long[!, :weight][i] = df_long[!, weight_columns[year_value]][i]
    end
end

In [ ]:
c = Between(:SAMPLING_PANEL_WEIGHT_1997, :SAMPLING_PANEL_WEIGHT_2021)
df_long = select(df_long, Not(c))


In [ ]:

#translate stata to julia: tsset PUBID_1997 week
df_long = sort!(df_long, [:PUBID_1997, :week])


In [ ]:
using RCall
@rput df_long

In [ ]:
CSV.write("D:/PSID_SHELF/03_NLSY97/data/Long_Employment_NLSY97.csv", df_long)

In [ ]:
R"""
Wealth <- read.csv("D:/PSID_SHELF/03_NLSY97/data/WEALTH.csv")
"""

In [ ]:
R"""
Wealth <- Wealth %>%
  rename(
    PubID = R0000100,
    Sex = R0536300,
    BirthDate_Month = R0536401,
    BirthDate_Year = R0536402,
    NetWealth_1997 = R1204800,
    SampleType_1997 = R1235800,
    SampWeight_1997 = R1236201,
    RaceEthnicity = R1482600,
    NetWealth_1998 = R2563400,
    SampWeight_1998 = R2600401,
    NetWealth_1999 = R3885000,
    SampWeight_1999 = R3958501,
    NetWealth_2000 = R5464200,
    SampWeight_2000 = R5510700,
    NetWealth_2001 = R7227900,
    SampWeight_2001 = R7274300,
    NetWealth_2002 = S1541800,
    SampWeight_2002 = S1598200,
    NetWealth_2003 = S2011600,
    SampWeight_2003 = S2067100
  )
"""

In [ ]:
R"""
names(Wealth)
"""

In [ ]:
R"""
summary(Wealth)
"""

In [ ]:
R"""
library(tidyr)
library(dplyr)

long_Wealth <- Wealth %>%
pivot_longer(
  cols = matches("^(NetWealth_|SampWeight_).+"),
  names_to = c(".value", "year"),
  names_sep = "_"
) %>%
mutate(year = as.numeric(year)) 

head(long_Wealth)

"""

In [ ]:
R"""
library(dplyr)
library(tidyr)
library(ggplot2)

long_Wealth <- long_Wealth %>%
  mutate(across(c(NetWealth), 
                ~replace_na(., 0)))
"""

In [ ]:
R"""
library(dplyr)

# Directly multiply variables by SampWeight without grouping by ID_1979 if possible
long_Wealth <- long_Wealth %>%
  mutate(NetWealth_weighted = NetWealth * SampWeight) %>%
  replace_na(list(NetWealth_weighted = 0))

# Calculate weighted sums and means more efficiently
weighted_sums <- long_Wealth %>%
  group_by(year) %>%
  summarise(NetWealth_sum = sum(NetWealth_weighted, na.rm = TRUE),
            TotalWeight = sum(SampWeight, na.rm = TRUE))

weighted_means <- weighted_sums %>%
  mutate(NetWealth_mean = NetWealth_sum / TotalWeight)
  
print(weighted_means)
"""

In [ ]:
R"""
library(dplyr)
library(ggplot2)

# Define the directory for saving the plots
plots_directory <- "D:/PSID_SHELF/Karlanser/Karlanser/NLSY97/Graph"

# Filter out years with zero values for NetWealth_mean and plot
weighted_means %>%
  filter(NetWealth_mean != 0) %>%
  ggplot(aes(x = year, y = NetWealth_mean)) +
  geom_line(color = "darkred") +
  geom_point(color = "black") +
  labs(title = "Weighted Mean of Net Wealth Over Years",
       x = "Year", y = "Weighted Mean") -> plot_net_wealth
ggsave(paste0(plots_directory, "/NetWealth_mean_over_years.png"), plot_net_wealth, width = 8, height = 6, dpi = 300)

"""